In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import re

In [2]:
df=pd.read_csv('../data/ticket_classification.csv')

In [ ]:
df=df.dropna()

In [4]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase the text
    text = text.lower()
    
    # 2. Remove BOTH actual newlines and literal "\n" text strings
    text = text.replace("\n", " ").replace("\\n", " ").replace("\r", " ")
    
    # 3. Isolates punctuation marks with spaces
    text = re.sub(r"([.,!?():\"'])", r" \1 ", text) 
    
    # 4. Remove extra spaces and squash down to a single clean space
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [5]:
df['Full Text'] = df['subject'].apply(clean_text)+" "+df['body'].apply(clean_text)

# Let's see what a row looks like before and after
print("BEFORE:\n", df['body'].iloc[0])
print("\nAFTER:\n", df['Full Text'].iloc[0])

BEFORE:
 Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?

AFTER:
 account disruption dear customer support team , i am writing to report a significant problem with the centralized account management portal , which currently appears to be offline . this outage is blocking access to account settings , leading to substantial inconvenience . i have attempted to log in multiple times using different browsers and devices , but the issue persists . could you please provide an update on the outage status a

In [6]:
# Check the word counts of your combined text
word_counts = df['Full Text'].apply(lambda x: len(x.split()))
print(word_counts.describe(percentiles=[0.5, 0.75, 0.90, 0.95]))

count    28261.000000
mean        66.957362
std         35.154567
min          1.000000
50%         65.000000
75%         93.000000
90%        106.000000
95%        119.000000
max        331.000000
Name: Full Text, dtype: float64


In [7]:
# 2. Encode Target
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['queue'])
num_classes = len(label_encoder.classes_)

# 3. Build Token Vocabulary from Dataset
words = set()
for text in df['Full Text']:
    words.update(text.split())

# Reserve 0 for Padding, 1 for Unknown words (OOV)
vocab = {word: i+2 for i, word in enumerate(sorted(words))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

In [8]:
import gensim.downloader as api
# 1. Stream/Load the GloVe 100d model directly from the gensim API repository
print("Loading GloVe embeddings via Gensim API... (This streams directly into memory)")
glove_vectors = api.load("glove-wiki-gigaword-100") 
print("GloVe loaded successfully!")

# 2. Build your PyTorch embedding matrix using your existing 'vocab' dictionary
def build_matrix_with_gensim(glove_model, vocab, embedding_dim=100):
    vocab_size = len(vocab)
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    
    # Initialize <UNK> token randomly; leave <PAD> at index 0 as zeros
    embedding_matrix[1] = np.random.normal(scale=0.6, size=(embedding_dim,))
    
    for word, idx in vocab.items():
        if word in glove_model:
            embedding_matrix[idx] = glove_model[word]
        elif idx > 1:
            # If word is not in GloVe, point it to the <UNK> vector
            embedding_matrix[idx] = embedding_matrix[1]
            
    return torch.tensor(embedding_matrix, dtype=torch.float32)

# Generate your embedding weights
embedding_matrix = build_matrix_with_gensim(glove_vectors, vocab, embedding_dim=100)
print("Embedding matrix shape:", embedding_matrix.shape)

Loading GloVe embeddings via Gensim API... (This streams directly into memory)
GloVe loaded successfully!
Embedding matrix shape: torch.Size([6695, 100])


In [9]:
MAX_LEN = 125 # Adjust based on your typical ticket length

def text_to_indices(text, vocab, max_len):
    tokens = text.split()
    indices = [vocab.get(token, 1) for token in tokens] # 1 is <UNK>
    if len(indices) < max_len:
        indices += [0] * (max_len - len(indices)) # 0 is <PAD>
    else:
        indices = indices[:max_len]
    return indices

# Generate indexed sequences
X_sequences = np.array([text_to_indices(txt, vocab, MAX_LEN) for txt in df['Full Text']])
y_labels = df['label'].values

# Train/Test Split
X_train, X_val, y_train, y_val = train_test_split(X_sequences, y_labels, test_size=0.2, random_state=42)

class TicketDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

train_loader = DataLoader(TicketDataset(X_train, y_train), batch_size=16, shuffle=True)
val_loader = DataLoader(TicketDataset(X_val, y_val), batch_size=32, shuffle=False)

In [10]:
class BiLSTMTicketClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, embedding_matrix, dropout_rate=0.5):
        super(BiLSTMTicketClassifier, self).__init__()
        
        # Load pre-trained GloVe weights
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True)
        
        # Bidirectional LSTM
        self.lstm = nn.Module()
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=dropout_rate if num_classes > 1 else 0.0
        )
        
        # Regularization Dropout
        self.dropout = nn.Dropout(dropout_rate)
        
        # Fully Connected Layer: hidden_dim * 2 because it's bidirectional
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, text):
        # text shape: [batch_size, seq_len]
        embedded = self.embedding(text) # shape: [batch_size, seq_len, embedding_dim]
        
        # lstm_out shape: [batch_size, seq_len, hidden_dim * 2]
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        # Max-pooling over time (extracts the most important features across the sequence)
        pooled_out, _ = torch.max(lstm_out, dim=1) # shape: [batch_size, hidden_dim * 2]
        
        out = self.dropout(pooled_out)
        logits = self.fc(out)
        return logits

In [11]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = BiLSTMTicketClassifier(
    vocab_size=len(vocab), 
    embedding_dim=100, 
    hidden_dim=128, 
    num_classes=num_classes, 
    embedding_matrix=embedding_matrix,
    dropout_rate=0.5
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
early_stopping = EarlyStopping(patience=3)

epochs = 20
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for sequences, labels in train_loader:
        sequences, labels = sequences.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    # Validation phase
    model.eval()
    val_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for sequences, labels in val_loader:
            sequences, labels = sequences.to(device), labels.to(device)
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    early_stopping(avg_val_loss)
    if early_stopping.early_stop:
        print("Early stopping triggered. Training stopped.")
        break

# Print final PyTorch Metrics
print("\n=== PyTorch BiLSTM Performance ===")
print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_))